ACROSS-PARTICIPANT RSA

In [1]:
import os
import numpy as np
from nilearn import image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, zscore, ttest_rel, pearsonr, ttest_ind
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
import seaborn as sns
from scipy.stats import sem
from scipy.signal import detrend
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from scipy.stats import ttest_1samp


In [2]:
foundationScores_dir = "/Users/silvycollin/Documents/GitHub/wholeBrain_narratives_MAC_MFT/text/foundationScores/"

In [3]:
whichVersion = 'persentence' #persentence, 3sec, 6sec

whichFoundationScore = "MAC_a_moral_nonmoral"

# save
save_dir = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/RSA_relToMoralnonmoral/"
)

In [4]:
# ------------------------
# Load 
# ------------------------

whichROI = "TPJ_refined_mask" # hippocampus,STG_MTG_refined_mask,TPJ_refined_mask

whichStory = "21styear"  # tunnel, 21styear

base_dir = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_first_level_per_sentence/"
)

if whichVersion =='persentence':
    foundationScores_file = f"{foundationScores_dir}{whichStory}_MFT_MAC.xlsx"
    foundationScores = pd.read_excel(foundationScores_file)

    # Construct the filename
    file_path = str(save_dir) + f"{whichVersion}_{whichStory}_RDM_fast_embeddings.csv"
    
    # Load it back as a pandas DataFrame
    RDM_df = pd.read_csv(file_path, index_col=0)  # index_col=0 if you saved the index
    
elif whichVersion =='3sec':
    print('3')
elif whichVersion =='6sec':
    print('6')
    
print(whichStory)
print(whichROI)

data_dir = f"{base_dir}{whichStory}/"

left = np.load(data_dir + f"{whichStory}_{whichROI}_L_3D_clean.npy")
right = np.load(data_dir + f"{whichStory}_{whichROI}_R_3D_clean.npy")

21styear
TPJ_refined_mask


In [5]:
# ------------------------
# load foundation scores 
# ------------------------
# start and end column is start and end of the sentences in seconds, TR was 1.5 sec

foundationScoresSelect = foundationScores[["MAC_a_moral_nonmoral","start","end"]]
foundationScoresSelect = foundationScoresSelect.copy()

# go from seconds to TRs for start and end column
foundationScoresSelect["startTR"] = foundationScoresSelect["start"] / 1.5
foundationScoresSelect["endTR"] = foundationScoresSelect["end"] / 1.5

# round to full TRs/volumes
foundationScoresSelect["startTR"] = foundationScoresSelect["startTR"].round(0).astype(int)
foundationScoresSelect["endTR"] = foundationScoresSelect["endTR"].round(0).astype(int)

# shift by 3 TRs for the HRF
foundationScoresSelect['startTR'] = foundationScoresSelect['startTR'] + 3
foundationScoresSelect['endTR'] = foundationScoresSelect['endTR'] + 3


In [6]:
# select only moral sentences
foundationScoresSelect_nonzero = foundationScoresSelect[foundationScoresSelect[whichFoundationScore] != 0]


In [7]:
foundationScoresSelect_nonzero

,MAC_a_moral_nonmoral,start,end,startTR,endTR
1,0.153846,24.138,32.755,19,25
2,0.500000,33.196,36.703,25,27
3,0.500000,36.683,40.370,27,30
5,1.000000,42.113,43.596,31,32
6,0.333333,44.277,46.822,33,34
...,...,...,...,...,...
1136,0.666667,3336.469,3338.832,2227,2229
1138,0.333333,3340.775,3342.877,2230,2232
1139,0.400000,3343.738,3347.883,2232,2235
1140,1.500000,3349.045,3352.048,2236,2238


In [8]:

# extract the column as a numpy array
vals = foundationScoresSelect_nonzero["MAC_a_moral_nonmoral"].values

# compute all pairwise absolute differences
RDM_moralscores = np.abs(vals[:, None] - vals[None, :])

# if you prefer a DataFrame:
RDM_moralscores = pd.DataFrame(RDM_moralscores,
                               index=foundationScoresSelect_nonzero.index,
                               columns=foundationScoresSelect_nonzero.index)


In [9]:

df = foundationScoresSelect_nonzero.copy()
E = len(df)
V, T, P = left.shape

# ------------------------------------------------------------
# 1. Extract event patterns
# ------------------------------------------------------------
def extract_patterns(hemisphere_array):
    V = hemisphere_array.shape[0]   # take voxel count from this hemisphere
    patterns = np.zeros((E, P, V))
    
    for e, (s, eTR) in enumerate(zip(df.startTR, df.endTR)):
        # average across TRs → (V, P), then transpose → (P, V)
        event_data = hemisphere_array[:, s:eTR+1, :].mean(axis=1).T
        patterns[e, :, :] = event_data

    return patterns

patterns_left  = extract_patterns(left)
patterns_right = extract_patterns(right)

# ------------------------------------------------------------
# 2. Compute leave-one-participant-out average patterns
# ------------------------------------------------------------
def compute_group_avg(patterns):
    E, P, V = patterns.shape
    avg = np.zeros((E, P, V))

    for p in range(P):
        others = np.delete(patterns, p, axis=1)  # shape = (E, P-1, V)
        avg[:, p, :] = others.mean(axis=1)
    return avg

avg_left  = compute_group_avg(patterns_left)
avg_right = compute_group_avg(patterns_right)

# ------------------------------------------------------------
# 3. Compute ISPC correlation RDMs
# ------------------------------------------------------------

# for every event pair (i, j), correlations are computed for every participant.

def compute_RDM(patterns, avg):
    E, P, V = patterns.shape
    RDM = np.zeros((E, E))

    for i in range(E):
        for j in range(E):
            corrs = []
            for p in range(P):
                x = patterns[i, p]
                y = avg[j, p]
                # correlation
                r = np.corrcoef(x, y)[0,1]
                corrs.append(r)
            RDM[i, j] = 1 - np.nanmean(corrs)  # dissimilarity
    return RDM

RDM_TPJ_left  = compute_RDM(patterns_left,  avg_left)
RDM_TPJ_right = compute_RDM(patterns_right, avg_right)



def compute_RDM_with_participant_corrs(patterns, avg):
    E, P, V = patterns.shape

    RDM = np.zeros((E, E))
    corrs_all = np.zeros((E, E, P))   # store each participant’s correlations

    for i in range(E):
        for j in range(E):
            for p in range(P):
                x = patterns[i, p]
                y = avg[j, p]
                r = np.corrcoef(x, y)[0,1]
                corrs_all[i, j, p] = r

            RDM[i, j] = 1 - np.nanmean(corrs_all[i, j, :])

    return RDM, corrs_all



# ------------------------------------------------------------
# 4. Correlate with BERT-score RDM
# ------------------------------------------------------------
# Extract upper triangle (excluding diagonal)
mask = np.triu(np.ones(RDM_df.shape), k=1).astype(bool)

bert_vec = RDM_df.values[mask]
left_vec  = RDM_TPJ_left[mask]
right_vec = RDM_TPJ_right[mask]

rho_left,  p_left  = spearmanr(bert_vec, left_vec)
rho_right, p_right = spearmanr(bert_vec, right_vec)

print("Left TPJ → behav RDM:")
print("Spearman rho =", rho_left, "; p =", p_left)

print("Right TPJ → behav RDM:")
print("Spearman rho =", rho_right, "; p =", p_right)


Left TPJ → behav RDM:
Spearman rho = 0.008888151393907442 ; p = 2.511799627622786e-07
Right TPJ → behav RDM:
Spearman rho = 0.007323140269583057 ; p = 2.1493012480672856e-05


## run models hierarchically with BERT and moral scores to determine unique variance explained

In [10]:
# mask2 = np.triu(np.ones(RDM_moralscores.shape), k=1).astype(bool)
# moral_vec = RDM_moralscores.values[mask2]

# X = np.column_stack([moral_vec, bert_vec])
# # z-score predictors

# X = StandardScaler().fit_transform(X)

# right_vec = (right_vec - right_vec.mean()) / right_vec.std()


# lr = LinearRegression().fit(X, right_vec)
# beta = lr.coef_
# print("Beta coefficients RIGHT (moral, BERT):", beta)


In [11]:
def compute_per_participant_RDMs(patterns):
    E, P, V = patterns.shape
    RDMs = np.zeros((P, E, E))

    for p in range(P):
        for i in range(E):
            for j in range(E):
                x = patterns[i, p]
                y = patterns[j, p]
                r = np.corrcoef(x, y)[0, 1]
                RDMs[p, i, j] = 1 - r  # dissimilarity
    return RDMs

In [12]:
RDMs_pp_right = compute_per_participant_RDMs(patterns_right)


In [13]:
# right_vecs = RDMs_pp_right

# n_subjects = right_vecs.shape[0]

# # Flatten upper triangle of moral RDM once
# mask = np.triu(np.ones((len(RDM_moralscores), len(RDM_moralscores))), k=1).astype(bool)
# moral_vec = RDM_moralscores.values[mask]

# # Stack with BERT predictor
# X = np.column_stack([moral_vec, bert_vec])
# X = StandardScaler().fit_transform(X)

# # Store per-subject betas
# betas = np.zeros((n_subjects, 2))  # columns: moral, BERT

# for i in range(n_subjects):
#     subj_rdm = right_vecs[i]  # 317 x 317
#     y = subj_rdm[mask]  # vectorize upper triangle
#     y = (y - y.mean()) / y.std()
#     lr = LinearRegression().fit(X, y)
#     betas[i, :] = lr.coef_

# # Compute mean ± SEM
# mean_betas = betas.mean(axis=0)
# sem_betas = betas.std(axis=0) / np.sqrt(n_subjects)

# # One-sample t-test across subjects
# tvals, pvals = ttest_1samp(betas, 0, axis=0)



## hierarchical models

In [ ]:

# -----------------------------------------------------------
# Setup: vectorize RDM and prepare predictors
# -----------------------------------------------------------
right_vecs = RDMs_pp_right
n_subjects = right_vecs.shape[0]

mask = np.triu(np.ones((len(RDM_moralscores), len(RDM_moralscores))), k=1).astype(bool)

moral_vec = RDM_moralscores.values[mask]
bert_vec_flat = bert_vec 

# Store ΔR² per subject
delta_R2_true = np.zeros(n_subjects)

# -----------------------------------------------------------
# Compute true ΔR² for each subject
# -----------------------------------------------------------
for s in range(n_subjects):

    y = right_vecs[s][mask]
    y = (y - y.mean()) / y.std()

    # first model: BERT only
    X_reduced = StandardScaler().fit_transform(bert_vec_flat.reshape(-1, 1))
    lr_reduced = LinearRegression().fit(X_reduced, y)
    R2_reduced = lr_reduced.score(X_reduced, y)

    # second model: BERT + MORAL
    X_full = np.column_stack([bert_vec_flat, moral_vec])
    X_full = StandardScaler().fit_transform(X_full)
    lr_full = LinearRegression().fit(X_full, y)
    R2_full = lr_full.score(X_full, y)

    delta_R2_true[s] = R2_full - R2_reduced

# Group-level true ΔR²
delta_R2_group_true = delta_R2_true.mean()

print("True mean ΔR² (adding moral above BERT):", delta_R2_group_true)

# -----------------------------------------------------------
# Permutation test
# -----------------------------------------------------------
n_perm = 10000
delta_R2_perm = np.zeros(n_perm)

for p in range(n_perm):

    # Permute MORAL predictor only
    moral_perm = np.random.permutation(moral_vec)

    delta_R2_subj = np.zeros(n_subjects)

    for s in range(n_subjects):

        y = right_vecs[s][mask]
        y = (y - y.mean()) / y.std()

        # Reduced model: BERT only
        X_reduced = StandardScaler().fit_transform(bert_vec_flat.reshape(-1, 1))
        R2_reduced = LinearRegression().fit(X_reduced, y).score(X_reduced, y)

        # Full model with permuted MORAL
        X_full_perm = np.column_stack([bert_vec_flat, moral_perm])
        X_full_perm = StandardScaler().fit_transform(X_full_perm)
        R2_full_perm = LinearRegression().fit(X_full_perm, y).score(X_full_perm, y)

        delta_R2_subj[s] = R2_full_perm - R2_reduced

    delta_R2_perm[p] = delta_R2_subj.mean()

# One-sided p-value (positive ΔR²)
p_value = np.mean(delta_R2_perm >= delta_R2_group_true)

print("Hierarchical permutation p-value (ΔR² > 0):", p_value)


True mean ΔR² (adding moral above BERT): 6.102050664869019e-06


In [ ]:
# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------
# delta_R2_perm       -> numpy array of permuted ΔR² (shape: n_perm,)
# delta_R2_group_true -> true mean ΔR² from data (single number)
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

# Histogram of permutation null distribution
counts, bins, patches = plt.hist(delta_R2_perm, bins=50, color='skyblue', edgecolor='black')

# Shade area >= true ΔR² (one-sided)
for patch, bin_edge in zip(patches, bins):
    if bin_edge >= delta_R2_group_true:
        patch.set_facecolor('orange')

# Plot true ΔR²
plt.axvline(delta_R2_group_true, color='red', linewidth=2, label=f'True ΔR² = {delta_R2_group_true:.6f}')

# Labels, title, legend
plt.xlabel("ΔR² under permutation")
plt.ylabel("Frequency")
plt.title("Permutation Null Distribution of ΔR²\n(Moral RDM adding variance above BERT)")
plt.legend()
plt.tight_layout()
plt.savefig(str(save_dir)+'/'+f"hierarchicalModel_moralAboveBert_{whichROI}_{whichStory}_{whichVersion}")
plt.show()
